# Module 1: Language Detection Pipeline

This notebook builds and evaluates a multi-class language identification classifier using TF-IDF feature extraction with character n-grams on the `papluca/language-identification` benchmark dataset.

In [ ]:
import unicodedata
import re
import pickle
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

## 1. Load Dataset

In [ ]:
dataset_train = load_dataset('papluca/language-identification', split='train')
dataset_val = load_dataset('papluca/language-identification', split='validation')
dataset_test = load_dataset('papluca/language-identification', split='test')

train_df = pd.DataFrame(dataset_train)
val_df = pd.DataFrame(dataset_val)
test_df = pd.DataFrame(dataset_test)

print('Train size:', len(train_df))
print('Validation size:', len(val_df))
print('Test size:', len(test_df))
print('Supported Languages:', train_df['labels'].unique())

## 2. Text Normalization and Preprocessing

In [ ]:
def normalize_text(text):
    if not isinstance(text, str):
        text = str(text)
    text = unicodedata.normalize('NFKC', text)
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text

train_df['clean_text'] = train_df['text'].apply(normalize_text)
test_df['clean_text'] = test_df['text'].apply(normalize_text)

## 3. Pipeline Construction and Training

In [ ]:
lang_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(2, 5),
        max_features=50000,
        sublinear_tf=True
    )),
    ('classifier', LogisticRegression(
        C=5.0,
        max_iter=1000,
        solver='lbfgs'
    ))
])

lang_pipeline.fit(train_df['clean_text'], train_df['labels'])

## 4. Model Evaluation

In [ ]:
preds = lang_pipeline.predict(test_df['clean_text'])
accuracy = accuracy_score(test_df['labels'], preds)
print(f'Test Set Accuracy: {accuracy * 100:.2f}%\n')
print(classification_report(test_df['labels'], preds))

## 5. Sample Inference Across Languages

In [ ]:
sample_queries = [
    'Where is my order package right now?',
    'Bonjour, je voudrais retourner mon colis endommage.',
    'Guten Tag, ich mochte meine Bestellung stornieren.',
    'Hola, cuando llega mi paquete a mi direccion?',
    'Vorrei chiedere un rimborso per il mio acquisto.'
]

for query in sample_queries:
    clean = normalize_text(query)
    probs = lang_pipeline.predict_proba([clean])[0]
    top_idx = probs.argmax()
    predicted_lang = lang_pipeline.classes_[top_idx]
    confidence = probs[top_idx]
    print(f'Query: "{query}"')
    print(f'Detected Language: {predicted_lang} (Confidence: {confidence * 100:.2f}%)\n')